# Signalement des anomalies d'adresses email — PM FINESS+

Les personnes morales portent désormais leurs coordonnées nativement, par la
liaison contact puis `Telecom`. Le référentiel externe d'emails que l'ancienne
chaîne devait joindre n'est plus nécessaire.

## 1. Imports et connexion

In [1]:
%run ../../config/config.ipynb

import sys, os
sys.path.insert(0, os.path.abspath('../..'))

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
from pathlib import Path

from src.chargement import (
    resoudre_tables, charger_telecoms, charger_ege, charger_pm,
)
from src.email_checker import (
    analyser_emails, marquer_doublons, evaluer_correspondance,
    charger_tlds_iana, charger_bases_noms,
)

pd.set_option("display.max_colwidth", 80)
print("Imports OK")


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
Connexion OK
Imports OK


## 2. Référentiels externes

In [2]:
DOSSIER_REF = Path("../../data/referentiels")

charger_tlds_iana(cache=DOSSIER_REF / "tlds-alpha-by-domain.txt")

df_prenoms    = pd.read_csv(DOSSIER_REF / "prenom.csv",     sep=",", dtype=str)
df_patronymes = pd.read_csv(DOSSIER_REF / "patronymes.csv", sep=",", dtype=str)

charger_bases_noms(df_prenoms, df_patronymes)

from src.email_checker import PRENOMS, PATRONYMES, TLDS_VALIDES, SEUIL_PRENOM, SEUIL_PATRONYME
print(f"TLD valides                      : {len(TLDS_VALIDES):,}")
print(f"Prénoms retenus (sum >= {SEUIL_PRENOM})     : {len(PRENOMS):,}")
print(f"Patronymes retenus (count >= {SEUIL_PATRONYME}) : {len(PATRONYMES):,}")

TLD valides                      : 1,437
Prénoms retenus (sum >= 500)     : 1,309
Patronymes retenus (count >= 100) : 14,710


In [3]:
df_communes = pd.read_csv(DOSSIER_REF / "communes-france-2026.csv", dtype=str)

from src.email_checker import charger_geo_insee
stats_geo = charger_geo_insee(df_communes, seuil_population=2000)
print(f"Villes retenues (pop >= 2000) : {stats_geo['villes']:,}")
print(f"Départements : {stats_geo['departements']} · Régions : {stats_geo['regions']}")

Villes retenues (pop >= 2000) : 5,200
Départements : 101 · Régions : 18


## 3. Chargement des PM FINESS+

In [4]:
TABLES = resoudre_tables(conn)

print("\nTelecoms :")
telecoms = charger_telecoms(conn, TABLES)

print("\nPM :")
df_pm = charger_pm(conn, TABLES, telecoms)

n_avec = df_pm["TEL_MAIL"].notna().sum()
print(f"\nPM FINESS actives : {len(df_pm):,}")
print(f"  avec email      : {n_avec:,}  ({n_avec/len(df_pm)*100:.1f} %)")
print(f"  sans email      : {len(df_pm) - n_avec:,}")

   pm           -> T_FAC_PmSmsse
   ege          -> T_FAC_Ege
   adresse      -> T_FAC_Adresse
   telecom      -> T_FAC_Telecom
   pm_adresse   -> T_FAC_PmSmsseAdresse
   ege_adresse  -> T_FAC_EgeAdresse
   pm_contact   -> T_FAC_PmSmsseContact
   ege_contact  -> T_FAC_EgeContact

Telecoms :
   3 lignes de canal sentinelle ecartees
   324 doublons (contact, canal) : premiere valeur conservee
   telecoms pivotes : 355,904 contacts, colonnes ['TEL_MAIL', 'TEL_TELECOPIE', 'TEL_TELEPHONE']

PM :
   84,015 lignes brutes
   84,015 lignes -> 56,194 structures, 3 contacts au maximum pour une structure

PM FINESS actives : 56,194
  avec email      : 29,642  (52.7 %)
  sans email      : 26,552


## 4. Détection des anomalies

In [5]:
print("Analyse des anomalies + classification...")
df = analyser_emails(df_pm, col_email="TEL_MAIL")
df = marquer_doublons(df, col_email_norm="email_norm", col_id="NB_PmSmsseId")
print(f"Analyse terminée — {len(df):,} PM traitées.")

Analyse des anomalies + classification...
Analyse terminée — 56,194 PM traitées.


## 5. Correspondance partie locale ↔ raison sociale / adresse

In [6]:
def _corresp_ligne(row):
    return evaluer_correspondance(
        local_part=row["local_part"],
        raison_principale=row.get("TX_DenominationPm", ""),
        raison_parent="",
        adresse=row.get("TX_LibelleVoie", ""),
        libelle_principal="(PM)",
        libelle_parent="",
    )

df["correspondance"] = ""
masque = df["local_part"] != ""
df.loc[masque, "correspondance"] = df.loc[masque].apply(_corresp_ligne, axis=1)

print("Répartition des correspondances :")
print(df[df["correspondance"] != ""]["correspondance"].value_counts().to_string())

Répartition des correspondances :
correspondance
Aucune correspondance            15805
Raison sociale (PM)              12877
Raison sociale (PM) + Adresse      772
Adresse                            188


In [7]:
from src.email_checker import detecter_geo

geo = df.apply(lambda row: detecter_geo(row["local_part"], row.get("TX_CogCommune")), axis=1)
df["geo_local"]      = geo.apply(lambda x: x["geo_local"])
df["geo_concordant"] = geo.apply(lambda x: x["geo_concordant"])

apercu = df[(df["geo_local"] != "") & (df["geo_concordant"] == False)]
print(f"Emails portant un lieu : {(df['geo_local'] != '').sum():,}")
print(f"  dont concordants     : {(df['geo_concordant'] == True).sum():,}")
print(f"  dont NON concordants : {len(apercu):,}")
apercu[["TX_DenominationPm", "TEL_MAIL", "geo_local"]].head(10)

Emails portant un lieu : 3,156
  dont concordants     : 2,384
  dont NON concordants : 772


,TX_DenominationPm,TEL_MAIL,geo_local
112,PHARMACIE LUZY-FOULON,contact.pharma.luzy@gmail.com,Ville (luzy)
125,PHARMACIE AIGUEPERSE,pharmacie.aigueperse@wanadoo.fr,Ville (aigueperse)
157,SELARL PHARMACIE SAVOYE,pharmacie2saintdenis@gmail.com,Ville (saintdenis)
158,PHARMACIE DAVIN,catherine.gorron@free.fr,Ville (gorron)
199,AIN DOMICILE SERVICES,ass.bourg@oranfe.fr,Ville (bourg)
239,MARPA LE JARDIN DES COURTANES,lejardindescourtanes@hotmail.fr,Ville (jardin)
264,A3H - AIDE ASSIST AUX AINES HANDICAPES,bourg01@agedorservices.com,Ville (bourg)
287,MAISON DE SANTE DE MIRIBEL,audreybrangers@yahoo.fr,Ville (angers)
298,LES JARDINS DE MONAL,lesjardinsdemonal@orange.fr,Ville (jardin)
411,SARL LES PEUPLIERS,bourg.direction@sofinam.fr,Ville (bourg)


## 6. Distribution globale des signalements

In [8]:
dist_niveau = df["niveau_anomalie"].value_counts().sort_index()
labels = {0: "Valides", 1: "Critiques (niveau 1)", 2: "Qualité (niveau 2)"}

print("─" * 55)
print("DISTRIBUTION GLOBALE DES SIGNALEMENTS")
print("─" * 55)
for niveau, n in dist_niveau.items():
    print(f"  {labels.get(niveau, f'Niveau {niveau}'):<30} : {n:>8,}  ({n/len(df)*100:5.1f} %)")
print("─" * 55)

print("\nDÉTAIL PAR TYPE D'ANOMALIE")
print("─" * 55)
detail = (
    df[df["niveau_anomalie"] > 0]
    .groupby(["niveau_anomalie", "code_anomalie"])
    .size().reset_index(name="nb")
    .sort_values(["niveau_anomalie", "nb"], ascending=[True, False])
)
for _, r in detail.iterrows():
    print(f"  [{r['niveau_anomalie']}] {r['code_anomalie']:<22} : {r['nb']:>6,}")

───────────────────────────────────────────────────────
DISTRIBUTION GLOBALE DES SIGNALEMENTS
───────────────────────────────────────────────────────
  Valides                        :   12,495  ( 22.2 %)
  Critiques (niveau 1)           :   26,640  ( 47.4 %)
  Qualité (niveau 2)             :   17,059  ( 30.4 %)
───────────────────────────────────────────────────────

DÉTAIL PAR TYPE D'ANOMALIE
───────────────────────────────────────────────────────
  [1] EMAIL_VIDE             : 26,552
  [1] EXTENSION_VIDE         :     25
  [1] DOMAINE_SANS_POINT     :     20
  [1] TLD_INEXISTANT         :     20
  [1] DOMAINE_INEXISTANT     :     15
  [1] ESPACE                 :      4
  [1] CARACTERE_INTERDIT     :      1
  [1] DOMAINE_VIDE           :      1
  [1] EXTENSION_TYPO         :      1
  [1] MANQUE_AROBAS          :      1
  [2] EMAIL_GRAND_PUBLIC     : 15,773
  [2] DOUBLON                :  1,286


## 7. Classification de la partie locale

In [9]:
classif = df[df["type_local"] != ""]
print(f"Classification de {len(classif):,} emails\n")

print("Partie locale :")
print(classif["type_local"].value_counts().to_string())
print("\nDomaine :")
print(classif["type_domaine"].value_counts().to_string())
print("\nCroisement local x domaine :")
print(pd.crosstab(classif["type_local"], classif["type_domaine"]).to_string())

Classification de 29,554 emails

Partie locale :
type_local
INDETERMINE          10396
INSTITUTIONNEL        8086
GENERIQUE             7310
NOMINATIF             2613
NOMINATIF_PARTIEL     1149

Domaine :
type_domaine
PUBLIC    15773
PROPRE    13781

Croisement local x domaine :
type_domaine       PROPRE  PUBLIC
type_local                       
GENERIQUE            6238    1072
INDETERMINE          3812    6584
INSTITUTIONNEL       1821    6265
NOMINATIF            1265    1348
NOMINATIF_PARTIEL     645     504


## 8. Anomalies critiques et signalements qualité

In [10]:
COLS_AFFICH = ["TX_NumFinessPm", "TX_DenominationPm", "TEL_MAIL",
               "code_anomalie", "libelle_anomalie", "suggestion"]
cols = [c for c in COLS_AFFICH if c in df.columns]

for niveau, titre in [(1, "ANOMALIES CRITIQUES"), (2, "SIGNALEMENTS QUALITÉ")]:
    sous = df[df["niveau_anomalie"] == niveau]
    print(f"\n{'='*60}")
    print(f"{titre} — {len(sous):,} cas")
    print(f"{'='*60}")
    for code_a, groupe in sous.groupby("code_anomalie"):
        print(f"\n  {code_a}  ({len(groupe):,} cas)")
        print(f"  {groupe['libelle_anomalie'].iloc[0]}")
        print(groupe[cols].head(5).to_string(index=False))


ANOMALIES CRITIQUES — 26,640 cas

  CARACTERE_INTERDIT  (1 cas)
  Caractère(s) interdit(s) dans l'adresse : ,
TX_NumFinessPm TX_DenominationPm                TEL_MAIL      code_anomalie                            libelle_anomalie suggestion
     640022125            PRISSM administratif@prissm,fr CARACTERE_INTERDIT Caractère(s) interdit(s) dans l'adresse : ,           

  DOMAINE_INEXISTANT  (15 cas)
  Le domaine « @gmail.fr » n'existe pas — variante invalide d'un service connu (ex : gmail.fr → gmail.com)
TX_NumFinessPm       TX_DenominationPm                       TEL_MAIL      code_anomalie                                                                                         libelle_anomalie suggestion
     020002804     PHARMACIE LAZZARANO    pharmacielazzarano@gmail.fr DOMAINE_INEXISTANT Le domaine « @gmail.fr » n'existe pas — variante invalide d'un service connu (ex : gmail.fr → gmail.com)           
     130050099    OJRG CARE4U SERVICES              ojrgfr71@gmail.fr DOMAINE_

## 9. Synthèse finale

In [11]:
n_total    = len(df)
n_vides    = (df["code_anomalie"] == "EMAIL_VIDE").sum()
n_renseign = max(n_total - n_vides, 1)

print("=" * 60)
print("SYNTHÈSE GLOBALE — SIGNALEMENT EMAIL PM FINESS")
print("=" * 60)
print(f"  PM actives analysées        : {n_total:>8,}")
print(f"  Email absent (vide)         : {n_vides:>8,}  ({n_vides/n_total*100:.1f} %)")
print()
print(f"  ── Sur les {n_renseign:,} emails renseignés ──")
print(f"  [1] Anomalies critiques     : {(df['niveau_anomalie'] == 1).sum():>8,}")
print(f"  [2] Signalements qualité    : {(df['niveau_anomalie'] == 2).sum():>8,}")
print(f"  [0] Emails valides          : {(df['niveau_anomalie'] == 0).sum():>8,}")
print("=" * 60)

SYNTHÈSE GLOBALE — SIGNALEMENT EMAIL PM FINESS
  PM actives analysées        :   56,194
  Email absent (vide)         :   26,552  (47.3 %)

  ── Sur les 29,642 emails renseignés ──
  [1] Anomalies critiques     :   26,640
  [2] Signalements qualité    :   17,059
  [0] Emails valides          :   12,495


## 10. Export Excel — Rapport de signalement

In [12]:
from openpyxl.styles import PatternFill, Font, Alignment

DOSSIER_SORTIE = Path("../../results/email")
DOSSIER_SORTIE.mkdir(parents=True, exist_ok=True)
FICHIER_SORTIE = DOSSIER_SORTIE / "signalement_emails_pm.xlsx"

COLS_EXPORT = [
    "NB_PmSmsseId", "TX_NumFinessPm", 
    "TX_DenominationPm", "TX_CogCommune", "departement", "adresse_complete",
    "TEL_MAIL", "email_norm",
    "code_anomalie", "libelle_anomalie", "suggestion",
    "type_local", "type_domaine", "correspondance",
    "geo_local", "geo_concordant",
]
cols_dispo = [c for c in COLS_EXPORT if c in df.columns]

def style_entete(ws, couleur):
    for cell in ws[1]:
        cell.fill = PatternFill("solid", start_color=couleur, end_color=couleur)
        cell.font = Font(bold=True, color="FFFFFF", name="Arial", size=10)
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
    ws.freeze_panes = "A2"
    ws.row_dimensions[1].height = 28

def remplir(ws, couleur):
    for row in ws.iter_rows(min_row=2):
        for cell in row:
            cell.fill = PatternFill("solid", start_color=couleur, end_color=couleur)
            cell.font = Font(name="Arial", size=9)

def auto_width(ws, max_w=50):
    for col in ws.columns:
        w = max((len(str(c.value or "")) for c in col), default=10)
        ws.column_dimensions[col[0].column_letter].width = min(w + 3, max_w)

df_vides     = df[df["code_anomalie"] == "EMAIL_VIDE"][cols_dispo]
df_critiques = df[(df["niveau_anomalie"] == 1) & (df["code_anomalie"] != "EMAIL_VIDE")][cols_dispo]
df_doublons  = df[df["code_anomalie"] == "DOUBLON"][cols_dispo]
df_grandpub  = df[df["code_anomalie"] == "EMAIL_GRAND_PUBLIC"][cols_dispo]
df_valides   = df[df["niveau_anomalie"] == 0][cols_dispo]

df_nominatifs   = df[df["type_local"].isin(["NOMINATIF", "NOMINATIF_PARTIEL"])][cols_dispo]
df_indetermines = df[df["type_local"] == "INDETERMINE"][cols_dispo]

df_synth = pd.DataFrame([
    {"Catégorie": "PM actifs total",     "Nombre": n_total,           "Part": "100 %"},
    {"Catégorie": "Email absent (vide)",         "Nombre": n_vides,           "Part": f"{n_vides/n_total*100:.1f} %"},
    {"Catégorie": "—",                           "Nombre": "",                "Part": ""},
    {"Catégorie": "[1] Anomalies critiques",     "Nombre": len(df_critiques), "Part": f"{len(df_critiques)/n_renseign*100:.1f} %"},
    {"Catégorie": "[2] Doublons",                "Nombre": len(df_doublons),  "Part": f"{len(df_doublons)/n_renseign*100:.1f} %"},
    {"Catégorie": "[2] Grand public",            "Nombre": len(df_grandpub),  "Part": f"{len(df_grandpub)/n_renseign*100:.1f} %"},
    {"Catégorie": "[0] Emails valides",          "Nombre": len(df_valides),   "Part": f"{len(df_valides)/n_renseign*100:.1f} %"},
    {"Catégorie": "—",                           "Nombre": "",                "Part": ""},
    {"Catégorie": "Vue croisée (hors total ci-dessus, quel que soit le domaine)", "Nombre": "", "Part": ""},
    {"Catégorie": "Nominatifs (NOMINATIF + PARTIEL)", "Nombre": len(df_nominatifs),   "Part": f"{len(df_nominatifs)/n_renseign*100:.1f} %"},
    {"Catégorie": "Indéterminés",                     "Nombre": len(df_indetermines), "Part": f"{len(df_indetermines)/n_renseign*100:.1f} %"},
])

with pd.ExcelWriter(FICHIER_SORTIE, engine="openpyxl") as writer:
    df_synth.to_excel(writer, sheet_name="Synthèse", index=False)
    style_entete(writer.sheets["Synthèse"], "1F3864")
    auto_width(writer.sheets["Synthèse"], max_w=40)

    for nom_f, donnees, ent, fond in [
        ("Emails_Vides",        df_vides,        "C62828", "F8CECC"),
        ("Anomalies_Critiques", df_critiques,    "C62828", "F8CECC"),
        ("Doublons",            df_doublons,     "F57C00", "FFF2CC"),
        ("Grand_Public",        df_grandpub,     "F57C00", "FFF2CC"),
        ("Emails_Valides",      df_valides,      "2E7D32", "EBF5EB"),
        ("Nominatifs",          df_nominatifs,   "1565C0", "D6E4F0"),
        ("Indetermines",        df_indetermines, "6A1B9A", "E8DAEF"),
    ]:
        if len(donnees) > 0:
            donnees.to_excel(writer, sheet_name=nom_f, index=False)
            style_entete(writer.sheets[nom_f], ent)
            remplir(writer.sheets[nom_f], fond)
            auto_width(writer.sheets[nom_f])

print(f"Export OK → {FICHIER_SORTIE.resolve()}")
print("\nFeuilles produites :")
for nom_f, donnees in [("Emails_Vides", df_vides), ("Anomalies_Critiques", df_critiques),
                       ("Doublons", df_doublons), ("Grand_Public", df_grandpub),
                       ("Emails_Valides", df_valides), ("Nominatifs", df_nominatifs),
                       ("Indetermines", df_indetermines)]:
    print(f"  {nom_f:<22} : {len(donnees):,}")

Export OK → /home/jovyan/work/finess+_refonte/signalement_data_contact_finess+/results/email/signalement_emails_pm.xlsx

Feuilles produites :
  Emails_Vides           : 26,552
  Anomalies_Critiques    : 88
  Doublons               : 1,286
  Grand_Public           : 15,773
  Emails_Valides         : 12,495
  Nominatifs             : 3,762
  Indetermines           : 10,396
